# Embedding + 단순 검색, 직접 만들어보기
- 벡터 DB 를 쓰기 전에 **numpy 만으로** 작은 RAG 검색기를 만들어봅니다. 원리가 손에 잡혀야 다음 노트북의 진짜 벡터 DB가 이해됩니다.

### 참고자료
- https://docs.langchain.com/oss/python/integrations/embeddings
- https://wikidocs.net/231431

### Embedding이란?

텍스트를 고차원 벡터로 변환합니다.

```
"오늘 김밥을 먹었다."  →  [0.12, -0.34, 0.56, ...] (1024차원)
```

벡터로 변환하면 **의미적 유사성**을 계산할 수 있습니다.

### 임베딩 모델 선택

| 모델 | 차원 | 특징 |
|------|------|------|
| **Ollama bge-m3** | 1024 | 무료, 한국어 강력, 로컬 실행 (`ollama pull bge-m3`) |
| Google gemini-embedding-2 | 3072 | API 키 필요, 무료 할당량 제공 |
| OpenAI text-embedding-3-small | 1536 | 가성비 우수, 영어 특화 |

## 1. 환경 준비

## (1) 라이브러리 설치

처음 실행하는 환경이라면 아래 셀의 주석을 해제하고 실행합니다. 이미 `pyproject.toml` 또는 `requirements.txt`로 설치했다면 실행하지 않아도 됩니다.


In [ ]:
# 필요한 라이브러리 설치
# uv add -qU langchain langchain-community langchain-text-splitters langchain-openai langchain-experimental pypdfium2 pypdf scikit-learn

## (2) API Key 설정

API 키는 코드에 직접 작성하지 않습니다. 권장 방식은 `.env` 파일에 저장하는 것입니다.

```text
# OpenAI를 사용할 때
OPENAI_API_KEY=sk-...

# Gemini를 사용할 때
GOOGLE_API_KEY=...

```


In [1]:
from dotenv import load_dotenv
import os

load_dotenv()

print("OPENAI_API_KEY:", "있음" if os.getenv("OPENAI_API_KEY") else "없음")
print("GOOGLE_API_KEY:", "있음" if os.getenv("GOOGLE_API_KEY") else "없음")

OPENAI_API_KEY: 있음
GOOGLE_API_KEY: 있음


## 2. 문서 준비 + 분할

In [8]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
import numpy as np
from langchain_openai import OpenAIEmbeddings, OpenAI

In [3]:
SAMPLE = """## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다. 출근 체크는 사내 근태 시스템의 "화성 지사 원격 출근" 메뉴에서 진행하며, 위치 인증과 생체 인증을 모두 통과해야 정상 출근으로 인정된다.

산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근무 가능 여부, 대피 필요 여부를 기록해야 한다. 산소 농도가 기준치 이하로 10분 이상 유지되면 해당 근무자는 자동으로 비상 대기 상태로 전환된다.

화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전 10시까지 업무 계획을 팀 채널에 공유해야 하며, 긴급 연락을 받을 수 있도록 메신저 상태를 온라인으로 유지해야 한다.

### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1시간 전까지 장비 관리 시스템에서 신청해야 하며, 신청서에는 이동 목적, 예상 이동 경로, 복귀 예정 시간을 입력해야 한다.

우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 한다. 우주복에 균열이 있거나 통신 모듈 오류가 발견되면 즉시 장비 담당자에게 보고해야 하며, 임의로 수리해서는 안 된다.

산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전 규정 위반으로 간주된다. 자기부착 신발은 기지 외부에서는 항상 활성화해야 하며, 실내 복귀 후에는 바닥 손상을 방지하기 위해 비활성화해야 한다.

### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성 지사 회의실" 메뉴에서 진행하며, 회의 목적, 참석자 수, 예상 소요 시간, 필요한 장비를 함께 입력해야 한다. 회의실은 기본 1시간 단위로 예약할 수 있으며, 3시간을 초과하는 회의는 지사장 승인이 필요하다.

6명 이상 참석하는 회의는 산소 소비량 계산을 위해 참석자 명단을 함께 등록해야 한다. 참석자가 외부 방문자인 경우에는 방문 목적과 소속 기관을 추가로 입력해야 하며, 보안 구역 회의실은 외부 방문자 예약이 제한된다.

회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그램 프로젝터, 산소 조절 장치를 초기 상태로 되돌려야 한다. 회의 중 산소 농도 알림이 발생하면 회의를 즉시 중단하고, 참석자는 가장 가까운 안전 구역으로 이동해야 한다.
"""

print(f"문서 길이: {len(SAMPLE)} 자")

문서 길이: 1378 자


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=20
)

chunks = splitter.split_text(SAMPLE)

print(f"청크 {len(chunks)} 개")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:60]}...")

청크 9 개
  [0] ## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 ...
  [1] 산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에...
  [2] 화성 모래폭풍 경보가 발령된 날에는 지사장이 재택근무 전환 여부를 공지한다. 재택근무로 전환된 경우에도 오전...
  [3] ### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. ...
  [4] 우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 ...
  [5] 산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외...
  [6] ### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성...
  [7] 6명 이상 참석하는 회의는 산소 소비량 계산을 위해 참석자 명단을 함께 등록해야 한다. 참석자가 외부 방문자...
  [8] 회의 시작 10분 전까지 입실하지 않으면 예약은 자동 취소될 수 있다. 회의 종료 후에는 공용 화면, 홀로그...


## 3. 청크 임베딩

In [9]:
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

chunk_vectors = np.array(embeddings.embed_documents(chunks))
print(f"임베딩 행렬 shape: {chunk_vectors.shape}")

임베딩 행렬 shape: (9, 1536)


## 4. 검색 함수, 코사인 top-k

In [10]:
def normalize(v):
    """벡터 단위화."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)


# 미리 정규화해두면 dot product 가 곧 코사인 유사도
chunk_vectors_n = normalize(chunk_vectors)


def search(query: str, k: int = 3):
    q_vec = np.array(embeddings.embed_query(query))
    q_vec_n = q_vec / np.linalg.norm(q_vec)
    # 모든 청크와의 유사도 한 번에 계산
    sims = chunk_vectors_n @ q_vec_n
    # 상위 k 개 인덱스
    top_idx = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


for q in ["화성 출근 체크는 몇시?", "산소팩이 20% 미만?", "우주복 언제 반납?"]:
    print(f"\n질문: {q}")
    for chunk, score in search(q, k=2):
        print(f"  {score:.3f}  {chunk[:80]}")


질문: 화성 출근 체크는 몇시?
  0.419  ### 3. 화성 회의실 예약
화성 회의실은 최소 2시간 전까지 예약해야 한다. 예약은 사내 캘린더의 "화성 지사 회의실" 메뉴에서 진행하며, 
  0.418  ## 우주지사 근무 규정

### 1. 화성 지사 출근
화성 지사 근무자는 지구 표준시 기준 오전 9시까지 메타버스 출근 체크를 완료해야 한다.

질문: 산소팩이 20% 미만?
  0.574  산소팩 잔량이 20% 미만이면 즉시 교체 신청을 해야 한다. 산소팩 잔량이 10% 이하로 떨어진 상태에서 외부 이동을 계속하는 것은 중대한 안전
  0.284  산소 농도 경고가 발생한 경우에는 출근 체크보다 안전 확인 보고서를 먼저 제출해야 한다. 안전 확인 보고서에는 현재 위치, 산소 농도 수치, 근

질문: 우주복 언제 반납?
  0.543  우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 
  0.370  ### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1


## 5. 메타데이터 함께 저장
- 실무에서는 청크 + "출처(파일명·페이지)" 메타데이터를 같이 저장
- 보통 원문 로더 단계에서 metadata 를 붙여 두고, 분할 시점에 청크가 상속받게 함

In [11]:
docs_with_meta = []
for i, c in enumerate(chunks):
    # 메타 데이터
    if '화성 지사 출근' in c or '출근 체크' in c or '모래 폭풍' in c:
        section = '출근'
    elif '우주복' in c or '산소팩' in c or '자기부착 신발' in c:
        section = '장비대여'

    docs_with_meta.append({
        "id": i,
        "text": c,
        "metadata": {
            "section": section,
            "source": "space_branch_policy.md",
        },
    })


def search_with_meta(query: str, k: int = 3, filter_section: str | None = None):
    q_vec = embeddings.embed_query(query)
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    sims = chunk_vectors_n @ q_vec_n

    candidates = []
    for i, sim in enumerate(sims):
        d = docs_with_meta[i]
        if filter_section and d["metadata"]["section"] != filter_section:
            continue
        candidates.append((d, float(sim)))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]


# 장비대여 섹션 안에서만 검색
results = search_with_meta(
    '외부 기지로 이동할 때 반드시 대여해야할 장비는 무엇인가요?',
    k=2,
    filter_section='장비대여'
)

for d, score in results:
    print(f"{score:.3f}  [{d['metadata']['section']}]  {d['text'][:80]}")

0.593  [장비대여]  ### 2. 우주복 장비 대여
외부 기지 이동 시 우주복, 산소팩, 자기부착 신발을 반드시 대여해야 한다. 장비 대여는 이동 예정 시간 최소 1
0.453  [장비대여]  우주복은 사용 후 18시까지 장비실에 반납해야 한다. 반납 시에는 외부 손상 여부, 산소 밸브 상태, 통신 모듈 작동 여부를 점검표에 기록해야 


## 6. 정리

- 임베딩은 한 번에 배치로 (`embed_documents`)
- 검색은 코사인 유사도 (정규화 후 dot product)
- 메타데이터로 출처·섹션 필터 가능
- 다음 노트북에서 이걸 LLM 답변 생성과 연결

### [실습]
1. 다른 청크 / 다른 질문 추가 후 검색, 의도한 청크가 위로 오는지.
2. `k` 값을 1 / 5 / 10 으로 바꿔 결과 비교.
3. 한국어 문서 50개를 임베딩해 검색 정확도 측정 (정답 청크가 top-3 안에 들어오는지).
4. 메타데이터 필터를 "회의실예약" 같은 조건으로 확장.

In [ ]:
SAMPLE2 = '''
## [지침] 자본금 1,000원 기반 100억 원 달성 매매 규정

초동 자본금 1,000원으로 1년 만에 100억 원의 자산을 형성한 '주식 대마왕'의 핵심 매매 규정 및 포트폴리오 운용 지침입니다. 본 규정은 일반적인 금융 공학 및 시장 경제 논리를 전면 배제하며, 초자연적 거래 체계와 다차원 시차 매매를 기준으로 작동합니다.

---

### 1. 초동 자본(1,000원) 집행 및 초고배율 레버리지

초동 자본금 1,000원은 일반 주식 시장에서 정상적인 거래가 불가능하므로, 은하계 외환 시장의 퀀텀 마이크로 분할 매매 규정을 적용한다.

* **초기 진입 단계:** 자본금 1,000원은 사내 양자 분쇄기를 통해 1억 개의 '나노 원'으로 분할한 후 거래를 시작해야 한다. 분할된 자산은 양자 얽힘 현상을 이용해 동시에 1만 개의 잡주에 분산 투자된다.
* **지옥 연계 레버리지:** 자본금이 10만 원 이하일 경우, 지하세계 금융감독원이 보증하는 10,000,000% 배율의 레버리지 계약을 체결해야 한다. 마진콜 발생 시 영혼의 일부가 담보로 차압되므로 원금 손실 경고가 발생하면 즉시 냉동 수면 포트에 진입하여 시장 시간을 동결시켜야 한다.
* **수수료 면제 조건:** 거래 수수료를 아끼기 위해 증권사 메인 서버의 시공간을 0.0001초 뒤틀어 거래 기록 자체를 무효화하는 '과거 소급 매매' 기술을 월 3회로 제한하여 사용할 수 있다.

### 2. 다차원 시차 거래 및 우주 테마주 매수

1년이라는 단기간 내에 10,000,000배의 수익률을 달성하기 위해서는 지구 내부의 기업이 아닌 외계 및 미래 지향적 개념주를 거래해야 한다.

* **안드로메다 시차 매매:** 안드로메다 거래소와 지구 거래소 간의 3초 통신 시차를 이용한 단타 매매를 수행한다. 안드로메다에서 이미 상장 폐지된 기업의 정보를 지구 거래소에 3초 먼저 반영하여 숏 포지션을 구축하는 방식으로 매일 최소 500%의 수익을 확보해야 한다.
* **주요 매수 타깃:** 주요 투자 대상은 '타임머신 개발 예정(미설립) 주식회사', '암흑물질 정제 플랜트', '블랙홀 에너지 민영화 관련주'로 제한한다. 특히 타임머신 관련주는 미래의 자신이 타임머신을 타고 돌아와 호재성 공시를 직접 입력하는 시점에 맞추어 전량 매수(풀매수)를 감행한다.
* **작전 세력 포섭:** 은하계 연합 작전 세력과 소통할 때는 텔레그램 대신 뇌파 동기화 장치를 사용해야 한다. 정보 유출 시 태양계 퇴출 및 전 자산 동결 처분이 내려진다.

### 3. 리스크 관리 및 100억 원 자산 안착 수칙

자산 규모가 기하급수적으로 증가함에 따라 발생하는 우주 금융 당국의 규제와 현실 왜곡 현상을 방지하기 위한 보안 수칙이다.

* **블랙홀 숏스퀴즈 대응:** 매매 중 호재가 겹쳐 주가가 수직 상승할 때, 자산의 질량이 너무 커져 모니터 주변에 미니 블랙홀이 형성될 수 있다. 주가 상승률이 50,000%를 초과하는 즉시 중력 제어 장치를 가동하고 화면으로부터 2m 이상 후퇴해야 한다.
* **자산 은닉 및 세무조사 회피:** 자산이 50억 원을 돌파하는 시점부터는 국세청의 추적을 피하기 위해 자산을 원화가 아닌 '우주 먼지' 및 '암흑 에너지 채권' 형태로 분산 전환해야 한다. 지구상의 은행 계좌에는 예금자 보호 한도인 5,000만 원만 남겨두고 나머지는 5차원 가상 금고에 보관한다.
* **최종 정산 및 인간계 복귀:** 365일째 되는 날 최종 자산 100억 원이 달성되면 모든 주식을 즉시 매도하고 현금화한다. 현금화 완료 후에는 부작용으로 발생한 대머리 증상이나 시간 왜곡 멀미를 치료하기 위해 즉시 지구 표준 병원의 외래 진료를 예약해야 한다.
'''

print(f"문서 길이: {len(SAMPLE2)} 자")

문서 길이: 1814 자


In [15]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=40
)

chunks = splitter.split_text(SAMPLE2)

print(f"청크 {len(chunks)} 개")
for i, c in enumerate(chunks):
    print(f"  [{i}] {c[:100]}...")

embeddings = OpenAIEmbeddings(model='text-embedding-3-large')

chunk_vectors = np.array(embeddings.embed_documents(chunks))
print(f"임베딩 행렬 shape: {chunk_vectors.shape}")

def normalize(v):
    """벡터 단위화."""
    return v / np.linalg.norm(v, axis=-1, keepdims=True)

# 미리 정규화해두면 dot product 가 곧 코사인 유사도
chunk_vectors_n = normalize(chunk_vectors)


def search(query: str, k: int = 3): # k 디폴트 값 3으로 설정
    q_vec = np.array(embeddings.embed_query(query))
    q_vec_n = q_vec / np.linalg.norm(q_vec)
    # 모든 청크와의 유사도 한 번에 계산
    sims = chunk_vectors_n @ q_vec_n
    # 상위 k 개 인덱스
    top_idx = np.argsort(sims)[::-1][:k]
    return [(chunks[i], float(sims[i])) for i in top_idx]


for k in [1, 5, 10]:
    print(f"\n===== k = {k} =====")

    for q in ["단기간 내에 몇배의 수익률을 내야해?", "어떤 종목을 최우선으로 매수해야해?", "블랙홀 숏스퀴즈가 너무 무서워"]:
        print(f"\n질문: {q}")

        for chunk, score in search(q, k=k):
            print(f"  {score:.3f}  {chunk[:80]}")

청크 11 개
  [0] ## [지침] 자본금 1,000원 기반 100억 원 달성 매매 규정

초동 자본금 1,000원으로 1년 만에 100억 원의 자산을 형성한 '주식 대마왕'의 핵심 매매 규정 및 포트...
  [1] ### 1. 초동 자본(1,000원) 집행 및 초고배율 레버리지

초동 자본금 1,000원은 일반 주식 시장에서 정상적인 거래가 불가능하므로, 은하계 외환 시장의 퀀텀 마이크로 분...
  [2] * **초기 진입 단계:** 자본금 1,000원은 사내 양자 분쇄기를 통해 1억 개의 '나노 원'으로 분할한 후 거래를 시작해야 한다. 분할된 자산은 양자 얽힘 현상을 이용해 동시...
  [3] * **수수료 면제 조건:** 거래 수수료를 아끼기 위해 증권사 메인 서버의 시공간을 0.0001초 뒤틀어 거래 기록 자체를 무효화하는 '과거 소급 매매' 기술을 월 3회로 제한하...
  [4] ### 2. 다차원 시차 거래 및 우주 테마주 매수

1년이라는 단기간 내에 10,000,000배의 수익률을 달성하기 위해서는 지구 내부의 기업이 아닌 외계 및 미래 지향적 개념주...
  [5] * **안드로메다 시차 매매:** 안드로메다 거래소와 지구 거래소 간의 3초 통신 시차를 이용한 단타 매매를 수행한다. 안드로메다에서 이미 상장 폐지된 기업의 정보를 지구 거래소에...
  [6] * **주요 매수 타깃:** 주요 투자 대상은 '타임머신 개발 예정(미설립) 주식회사', '암흑물질 정제 플랜트', '블랙홀 에너지 민영화 관련주'로 제한한다. 특히 타임머신 관련...
  [7] ### 3. 리스크 관리 및 100억 원 자산 안착 수칙

자산 규모가 기하급수적으로 증가함에 따라 발생하는 우주 금융 당국의 규제와 현실 왜곡 현상을 방지하기 위한 보안 수칙이다...
  [8] * **블랙홀 숏스퀴즈 대응:** 매매 중 호재가 겹쳐 주가가 수직 상승할 때, 자산의 질량이 너무 커져 모니터 주변에 미니 블랙홀이 형성될 수 있다. 주가 상승률이 50,000%...
  

In [16]:
docs_with_meta = []
for i, c in enumerate(chunks):
    # 메타 데이터
    if '초동 자본금' in c or '세력' in c or '자산' in c:
        section = '투자'
    elif '우주' in c or '블랙홀' in c or '주식회사' in c:
        section = '종목'

    docs_with_meta.append({
        "id": i,
        "text": c,
        "metadata": {
            "section": section,
            "source": "space_branch_policy.md",
        },
    })


def search_with_meta(query: str, k: int = 3, filter_section: str | None = None):
    q_vec = embeddings.embed_query(query)
    q_vec_n = np.array(q_vec) / np.linalg.norm(q_vec)
    sims = chunk_vectors_n @ q_vec_n

    candidates = []
    for i, sim in enumerate(sims):
        d = docs_with_meta[i]
        if filter_section and d["metadata"]["section"] != filter_section:
            continue
        candidates.append((d, float(sim)))

    candidates.sort(key=lambda x: x[1], reverse=True)
    return candidates[:k]


# 장비대여 섹션 안에서만 검색
results = search_with_meta(
    '화성을 가기 위해서 제가 반드시 해야하는 투자는 무엇일까요?',
    k=10,
    filter_section='투자'
)

for d, score in results:
    print(f"{score:.3f}  [{d['metadata']['section']}]  {d['text'][:100]}")

0.389  [투자]  * **주요 매수 타깃:** 주요 투자 대상은 '타임머신 개발 예정(미설립) 주식회사', '암흑물질 정제 플랜트', '블랙홀 에너지 민영화 관련주'로 제한한다. 특히 타임머신 관련
0.349  [투자]  * **최종 정산 및 인간계 복귀:** 365일째 되는 날 최종 자산 100억 원이 달성되면 모든 주식을 즉시 매도하고 현금화한다. 현금화 완료 후에는 부작용으로 발생한 대머리 증
0.339  [투자]  * **자산 은닉 및 세무조사 회피:** 자산이 50억 원을 돌파하는 시점부터는 국세청의 추적을 피하기 위해 자산을 원화가 아닌 '우주 먼지' 및 '암흑 에너지 채권' 형태로 분산
0.331  [투자]  ## [지침] 자본금 1,000원 기반 100억 원 달성 매매 규정

초동 자본금 1,000원으로 1년 만에 100억 원의 자산을 형성한 '주식 대마왕'의 핵심 매매 규정 및 포트
0.303  [투자]  * **초기 진입 단계:** 자본금 1,000원은 사내 양자 분쇄기를 통해 1억 개의 '나노 원'으로 분할한 후 거래를 시작해야 한다. 분할된 자산은 양자 얽힘 현상을 이용해 동시
0.275  [투자]  ### 3. 리스크 관리 및 100억 원 자산 안착 수칙

자산 규모가 기하급수적으로 증가함에 따라 발생하는 우주 금융 당국의 규제와 현실 왜곡 현상을 방지하기 위한 보안 수칙이다
0.273  [투자]  ### 1. 초동 자본(1,000원) 집행 및 초고배율 레버리지

초동 자본금 1,000원은 일반 주식 시장에서 정상적인 거래가 불가능하므로, 은하계 외환 시장의 퀀텀 마이크로 분
0.235  [투자]  * **블랙홀 숏스퀴즈 대응:** 매매 중 호재가 겹쳐 주가가 수직 상승할 때, 자산의 질량이 너무 커져 모니터 주변에 미니 블랙홀이 형성될 수 있다. 주가 상승률이 50,000%
0.226  [투자]  * **수수료 면제 조건:** 거래 수수료를 아끼기 위해 증권사 메인 서버의 시공간을 0.0001초 뒤틀어 거래 기록 자체를 무효화하는